# Gold Layer: Fact Sales Enriched (Holiday API Data)

**Pattern**: Streaming Enrichment + Metadata DQ  
**Sources**: 
- Gold Fact Sales (core transactions)
- Silver Holidays CDC (API-sourced enrichment data)

**Business Value**: Enrich core sales facts with external holiday data for seasonal analysis without polluting the base fact table.

**Holiday Flags Added**:
- `is_holiday` - Is this date a holiday?
- `holiday_name` - Name of the holiday
- `is_pre_holiday_week` - 7 days before holiday
- `is_post_holiday_week` - 7 days after holiday

### Step 0: Define Metadata (The 'Invisible' Rules)
Run this after the first execution to 'Tag' your columns with DQ rules.

In [0]:
%run "../utils/utils_quality_control"

In [0]:
target_table = "workspace.gold.fact_sales_enriched"

if spark.catalog.tableExists(target_table):
    spark.sql(f"ALTER TABLE {target_table} CHANGE COLUMN order_number COMMENT 'DQ: NOT NULL'")
    spark.sql(f"ALTER TABLE {target_table} CHANGE COLUMN sales_amount COMMENT 'DQ: >= 0'")
    print("✓ Enriched sales metadata (DQ Tags) updated in Catalog.")
else:
    print("ℹ Table does not exist yet. Run the stream first to create it.")

In [0]:
from pyspark.sql.functions import col, lit, when, datediff, month, dayofmonth, year, make_date, row_number, broadcast, max as spark_max
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# Configuration
target_table = "workspace.gold.fact_sales_enriched"
quarantine_table = "workspace.gold.quarantine_sales_enriched"

print("Loading fact_sales table...")
fact_sales_df = spark.table("workspace.gold.fact_sales")

print("Extracting month and day from order_date...")
df_with_dates = fact_sales_df.withColumn(
    "order_month",
    month(col("order_date"))
).withColumn(
    "order_day",
    dayofmonth(col("order_date"))
).withColumn(
    "order_year",
    year(col("order_date"))
)

print("Loading holidays data...")
holidays_df = spark.table("workspace.silver.holidays_cdc")

# Extract month and day from holiday dates for year-agnostic matching
holidays_with_month_day = holidays_df.withColumn(
    "holiday_month",
    month(col("date"))
).withColumn(
    "holiday_day",
    dayofmonth(col("date"))
)

print("Joining with holidays (year-agnostic: matching by month+day only)...")
enriched_df = df_with_dates.join(
    holidays_with_month_day,
    (col("order_month") == col("holiday_month")) & (col("order_day") == col("holiday_day")),
    "left"
).select(
    df_with_dates["*"],
    holidays_with_month_day["holiday_name"],
    when(col("holiday_name").isNotNull(), lit(True)).otherwise(lit(False)).alias("is_holiday")
)

print("Adding pre/post holiday week flags (year-agnostic)...")
# Get unique holiday month+day combinations
holiday_month_day_df = holidays_df.select(
    month(col("date")).alias("h_month"),
    dayofmonth(col("date")).alias("h_day")
).distinct()

# Cross join and construct synthetic holiday dates in the same year as each order
enriched_with_synthetic_holidays = enriched_df.crossJoin(
    broadcast(holiday_month_day_df)
).withColumn(
    "synthetic_holiday_date",
    make_date(col("order_year"), col("h_month"), col("h_day"))
).withColumn(
    "days_diff",
    datediff(col("synthetic_holiday_date"), col("order_date"))
)

# Aggregate to determine if ANY holiday is within the pre/post window
enriched_with_flags = enriched_with_synthetic_holidays.withColumn(
    "is_pre_week_flag",
    when((col("days_diff") >= 1) & (col("days_diff") <= 7), lit(True)).otherwise(lit(False))
).withColumn(
    "is_post_week_flag",
    when((col("days_diff") >= -7) & (col("days_diff") <= -1), lit(True)).otherwise(lit(False))
).groupBy([col(c) for c in enriched_df.columns]).agg(
    spark_max("is_pre_week_flag").alias("is_pre_holiday_week"),
    spark_max("is_post_week_flag").alias("is_post_holiday_week")
)

# Handle multiple holidays on same date (deduplication)
window_spec = Window.partitionBy("order_number", "product_key").orderBy(col("holiday_name"))
enriched_deduped = enriched_with_flags.withColumn(
    "row_num",
    row_number().over(window_spec)
).filter(col("row_num") == 1).drop("row_num")

print("Applying data quality checks...")
validated_df = apply_metadata_dq(enriched_deduped, target_table)

clean_df = validated_df.filter("is_invalid = false").drop("is_invalid", "dq_errors")
quarantine_df = validated_df.filter("is_invalid = true")

print("Saving data...")
if quarantine_df.count() > 0:
    print(f"⚠ {quarantine_df.count()} records quarantined")
    quarantine_df.write.format("delta").mode("append").saveAsTable(quarantine_table)

print("Overwriting enriched table...")
clean_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

print(f"✓ Enrichment complete. {clean_df.count()} records in {target_table}")

In [0]:
%sql
SELECT 
  COUNT(*) AS total_records,
  COUNT(CASE WHEN is_holiday THEN 1 END) AS holiday_records,
  COUNT(CASE WHEN is_pre_holiday_week THEN 1 END) AS pre_holiday_week_records,
  COUNT(CASE WHEN is_post_holiday_week THEN 1 END) AS post_holiday_week_records,
  ROUND(100.0 * COUNT(CASE WHEN is_holiday THEN 1 END) / COUNT(*), 2) AS holiday_pct
FROM workspace.gold.fact_sales_enriched;

In [0]:
%sql
SELECT *
FROM workspace.gold.fact_sales_enriched
WHERE is_holiday = TRUE
LIMIT 10;

In [0]:
%sql
-- Verify holiday enrichment by holiday name
SELECT 
  holiday_name,
  COUNT(*) AS sales_count,
  SUM(sales_amount) AS total_revenue,
  AVG(sales_amount) AS avg_order_value
FROM workspace.gold.fact_sales_enriched
WHERE is_holiday = TRUE
GROUP BY holiday_name
ORDER BY total_revenue DESC;